In [6]:
import pandas as pd
from pathlib import Path

DATA_PROCESSED = Path('../data/processed')
REPORTS = Path('../reports')
REPORTS.mkdir(exist_ok=True)

In [7]:
deals_dict = pd.DataFrame([
    ('Id', 'string', 'Уникальный идентификатор сделки в CRM (19-значное число)', 'непустая строка длиной 19 цифр', 'NaN — удалить строку (2 случая)', 'NaN в Id'),
    ('Deal Owner Name', 'string', 'Имя менеджера-владельца сделки', '27 уникальных значений (имена менеджеров)', 'NaN сохраняем (29 случаев)', 'не удаляем'),
    ('Closing Date', 'datetime', 'Дата завершения работы по сделке (закрытия — успешного или с потерей)', 'дата в диапазоне периода данных', 'NaN сохраняем (32% — открытые сделки)', 'не удаляем'),
    ('Quality', 'string', 'Качество лида по оценке менеджера', 'A-High, B-Medium, C-Low, D-Non Target, E-Non Qualified', 'F (3 случая) → NaN; NaN сохраняем', 'не удаляем'),
    ('Stage', 'string', 'Текущая стадия сделки в воронке продаж', '13 валидных значений: Lost, Payment Done, Call Delayed и др.', 'NaN сохраняем', 'не удаляем'),
    ('Lost Reason', 'string', 'Причина потери сделки (заполняется при Stage = Lost)', '22 уникальных значения', 'NaN сохраняем (для не-Lost сделок)', 'не удаляем'),
    ('Page', 'string', 'Страница сайта, с которой оставлена заявка', 'строка с путём страницы', 'NaN сохраняем', 'не удаляем'),
    ('Campaign', 'string', 'Рекламная кампания привлечения лида (UTM-параметр)', '155 уникальных значений', 'NaN сохраняем', 'не удаляем'),
    ('SLA', 'timedelta', 'Время реакции менеджера: от подачи заявки лидом до контакта продажника', 'неотрицательная длительность', 'NaN сохраняем; единичные SLA > 30 дней отмечаем как аномалии', 'не удаляем'),
    ('Content', 'string', 'UTM-параметр content (тип контента/баннера)', 'строка', 'NaN сохраняем', 'не удаляем'),
    ('Term', 'string', 'UTM-параметр term (ключевая фраза/таргетинг)', 'строка', 'NaN сохраняем', 'не удаляем'),
    ('Source', 'string', 'Маркетинговый источник привлечения лида', '13 значений: Facebook Ads, Google Ads, Organic, Test и др.', 'NaN сохраняем (2 случая)', 'не удаляем'),
    ('Payment Type', 'string', 'Тип оплаты курса', 'Recurring Payments (помесячно), One Payment (единоразово), Reservation (бронь места)', 'NaN сохраняем', 'не удаляем'),
    ('Product', 'string', 'Продукт обучения', 'Digital Marketing, UX/UI Design, Web Developer, Data Analytics, Find yourself in IT', 'NaN сохраняем (для лидов без выбранного продукта)', 'не удаляем'),
    ('Education Type', 'string', 'Формат расписания обучения', 'Morning (утром), Evening (вечером)', 'NaN сохраняем', 'не удаляем'),
    ('Created Time', 'datetime', 'Дата и время создания сделки в CRM', 'дата в диапазоне периода данных', 'NaN сохраняем (2 случая)', 'не удаляем'),
    ('Course duration', 'float', 'Общая продолжительность курса в месяцах', '6 или 11', 'NaN сохраняем', 'не удаляем'),
    ('Months of study', 'float', 'Текущий месяц обучения активного ученика на дату выгрузки', '0–11 (целое число)', 'NaN сохраняем (для неактивных учеников)', 'не удаляем'),
    ('Initial Amount Paid', 'float (EUR)', 'Сумма первого/текущего платежа клиента (может быть вся сумма обучения или её часть)', '0–11500', 'NaN сохраняем; нормализация европейского формата € X.XXX,YY', 'не удаляем'),
    ('Offer Total Amount', 'float (EUR)', 'Полная стоимость обучения по договору (сколько клиент заплатит, если доучится до конца и внесёт все платежи)', '0–11500', 'NaN сохраняем; нормализация европейского формата', 'не удаляем'),
    ('Contact Name', 'string', 'ID контакта в CRM (несмотря на название, это идентификатор, не имя)', '19-значное число как строка', 'NaN сохраняем (63 случая)', 'не удаляем'),
    ('City', 'string', 'Город проживания клиента', 'название города на немецком', '"-" (348 случаев) → NaN; NaN сохраняем', 'не удаляем'),
    ('Level of Deutsch', 'string', 'Уровень владения немецким языком (стандарт CEFR)', 'A1, A2, B1, B2, C1, C2', 'Нормализация: regex-экстрактор букв и цифр; кириллица → латиница; верхний регистр. Свободный текст без явного уровня → NaN', 'не удаляем'),
    ('Stage_Group', 'string', '[Производная] Группировка Stage для анализа воронки', 'Won, Lost, In Progress, Free', 'создаётся в чистке через mapping', 'не удаляем'),
    ('is_duplicate_lead', 'bool', '[Производная] Флаг дублирующего лида (Lost Reason = Duplicate)', 'True / False', 'создаётся в чистке', 'не удаляем'),
    ('is_outlier_contact', 'bool', '[Производная] Флаг сделки контакта-аутлайера (>5 сделок без единой Won)', 'True / False', 'создаётся в чистке', 'не удаляем'),
], columns=['Column', 'Type', 'Description', 'Valid values', 'Action on invalid', 'Deletion criteria'])

# Добавим автоматически количество пропусков из реальных данных
deals = pd.read_parquet(DATA_PROCESSED / 'deals_clean.parquet')
nulls = deals.isna().sum().reset_index()
nulls.columns = ['Column', 'Nulls']
nulls['Nulls %'] = (nulls['Nulls'] / len(deals) * 100).round(1)

deals_dict = deals_dict.merge(nulls, on='Column', how='left')
deals_dict

,Column,Type,Description,Valid values,Action on invalid,Deletion criteria,Nulls,Nulls %
0,Id,string,Уникальный идентификатор сделки в CRM (19-знач...,непустая строка длиной 19 цифр,NaN — удалить строку (2 случая),NaN в Id,0,0.0
1,Deal Owner Name,string,Имя менеджера-владельца сделки,27 уникальных значений (имена менеджеров),NaN сохраняем (29 случаев),не удаляем,0,0.0
2,Closing Date,datetime,Дата завершения работы по сделке (закрытия — у...,дата в диапазоне периода данных,NaN сохраняем (32% — открытые сделки),не удаляем,6948,32.2
3,Quality,string,Качество лида по оценке менеджера,"A-High, B-Medium, C-Low, D-Non Target, E-Non Q...",F (3 случая) → NaN; NaN сохраняем,не удаляем,2256,10.4
4,Stage,string,Текущая стадия сделки в воронке продаж,"13 валидных значений: Lost, Payment Done, Call...",NaN сохраняем,не удаляем,0,0.0
5,Lost Reason,string,Причина потери сделки (заполняется при Stage =...,22 уникальных значения,NaN сохраняем (для не-Lost сделок),не удаляем,5469,25.3
6,Page,string,"Страница сайта, с которой оставлена заявка",строка с путём страницы,NaN сохраняем,не удаляем,0,0.0
7,Campaign,string,Рекламная кампания привлечения лида (UTM-парам...,155 уникальных значений,NaN сохраняем,не удаляем,5526,25.6
8,SLA,timedelta,Время реакции менеджера: от подачи заявки лидо...,неотрицательная длительность,NaN сохраняем; единичные SLA > 30 дней отмечае...,не удаляем,6060,28.1
9,Content,string,UTM-параметр content (тип контента/баннера),строка,NaN сохраняем,не удаляем,7446,34.5


In [8]:
calls_dict = pd.DataFrame([
    ('Id', 'string', 'Уникальный идентификатор звонка в CRM', 'непустая строка-число', 'не встречалось NaN', 'не удаляем'),
    ('Call Start Time', 'datetime', 'Дата и время начала звонка', 'дата в диапазоне периода данных', 'не встречалось NaN', 'не удаляем'),
    ('Call Owner Name', 'string', 'Имя менеджера, совершившего/принявшего звонок', '33 уникальных значения (включая менеджеров не из Deals)', 'не встречалось NaN', 'не удаляем'),
    ('CONTACTID', 'string', 'ID контакта в CRM, к которому привязан звонок', '19-значное число как строка', 'NaN сохраняем (3933 случая — звонки без привязки к контакту)', 'не удаляем'),
    ('Call Type', 'string', 'Тип звонка', 'Outbound (исходящий), Inbound (входящий), Missed (пропущенный)', 'не встречалось NaN', 'не удаляем'),
    ('Call Duration (in seconds)', 'float', 'Продолжительность звонка в секундах', '≥ 0; 22% звонков = 0 (не дозвонился до гудка / сбросил)', 'NaN сохраняем (83 случая)', 'не удаляем'),
    ('Call Status', 'string', 'Статус звонка', '11 значений: Attended Dialled, Unattended Dialled, Missed, Received и др.', 'не встречалось NaN', 'не удаляем'),
    ('Outgoing Call Status', 'string', 'Статус исходящего звонка (только для Outbound)', 'Completed, Overdue, Cancelled, Scheduled', 'NaN сохраняем (8999 — соответствует Inbound + Missed по логике CRM)', 'не удаляем'),
    ('Scheduled in CRM', 'float', 'Флаг 0/1: был ли звонок запланирован в CRM', '0.0 или 1.0; для Inbound/Missed по определению NaN', 'NaN сохраняем (8999 случаев)', 'не удаляем'),
    ('is_attended', 'bool', '[Производная] Флаг "ответили на звонок"', 'True (Attended Dialled, Received, Scheduled Attended) / False (остальные)', 'создаётся в чистке через mapping', 'не удаляем'),
], columns=['Column', 'Type', 'Description', 'Valid values', 'Action on invalid', 'Deletion criteria'])

calls = pd.read_parquet(DATA_PROCESSED / 'calls_clean.parquet')
nulls_calls = calls.isna().sum().reset_index()
nulls_calls.columns = ['Column', 'Nulls']
nulls_calls['Nulls %'] = (nulls_calls['Nulls'] / len(calls) * 100).round(1)

calls_dict = calls_dict.merge(nulls_calls, on='Column', how='left')
calls_dict

,Column,Type,Description,Valid values,Action on invalid,Deletion criteria,Nulls,Nulls %
0,Id,string,Уникальный идентификатор звонка в CRM,непустая строка-число,не встречалось NaN,не удаляем,0,0.0
1,Call Start Time,datetime,Дата и время начала звонка,дата в диапазоне периода данных,не встречалось NaN,не удаляем,0,0.0
2,Call Owner Name,string,"Имя менеджера, совершившего/принявшего звонок",33 уникальных значения (включая менеджеров не ...,не встречалось NaN,не удаляем,0,0.0
3,CONTACTID,string,"ID контакта в CRM, к которому привязан звонок",19-значное число как строка,NaN сохраняем (3933 случая — звонки без привяз...,не удаляем,3933,4.1
4,Call Type,string,Тип звонка,"Outbound (исходящий), Inbound (входящий), Miss...",не встречалось NaN,не удаляем,0,0.0
5,Call Duration (in seconds),float,Продолжительность звонка в секундах,≥ 0; 22% звонков = 0 (не дозвонился до гудка /...,NaN сохраняем (83 случая),не удаляем,83,0.1
6,Call Status,string,Статус звонка,"11 значений: Attended Dialled, Unattended Dial...",не встречалось NaN,не удаляем,0,0.0
7,Outgoing Call Status,string,Статус исходящего звонка (только для Outbound),"Completed, Overdue, Cancelled, Scheduled",NaN сохраняем (8999 — соответствует Inbound + ...,не удаляем,8999,9.4
8,Scheduled in CRM,float,Флаг 0/1: был ли звонок запланирован в CRM,0.0 или 1.0; для Inbound/Missed по определению...,NaN сохраняем (8999 случаев),не удаляем,8999,9.4
9,is_attended,bool,"[Производная] Флаг ""ответили на звонок""","True (Attended Dialled, Received, Scheduled At...",создаётся в чистке через mapping,не удаляем,0,0.0


In [9]:
spend_dict = pd.DataFrame([
    ('Date', 'datetime', 'Дата расхода рекламного бюджета', 'дата в диапазоне 03.07.2023 – 21.06.2024', 'не встречалось NaN', 'не удаляем'),
    ('Source', 'string', 'Платный канал привлечения лидов', '14 значений: Facebook Ads, Tiktok Ads, Google Ads, Bloggers, Telegram posts, SMM, Webinar, Organic, CRM, Partnership, Test, Offline, Radio, Youtube Ads', 'не встречалось NaN', 'не удаляем'),
    ('Campaign', 'string', 'Кампания, на которую были израсходованы средства', '50 уникальных значений (имена кампаний)', 'NaN сохраняем (29% — каналы без кампанийной структуры: Organic, CRM, Webinar, Bloggers, Telegram posts, Partnership и др.)', 'не удаляем'),
    ('Impressions', 'int', 'Количество показов рекламы в указанную дату', '≥ 0; 27% записей = 0 (паузы кампаний)', 'не встречалось NaN', 'не удаляем'),
    ('Spend', 'float (EUR)', 'Расход на рекламу в евро', '≥ 0; общий бюджет за период €149 523', 'не встречалось NaN', 'не удаляем'),
    ('Clicks', 'int', 'Количество уникальных переходов по рекламной ссылке в указанную дату', '≥ 0', 'не встречалось NaN', 'не удаляем'),
    ('AdGroup', 'string', 'Название группы объявлений (только для платных таргетированных каналов)', '24 уникальных значения', 'NaN сохраняем (33% — каналы без AdGroup структуры)', 'не удаляем'),
    ('Ad', 'string', 'Название рекламного объявления', '176 уникальных значений', 'NaN сохраняем (33%)', 'не удаляем'),
], columns=['Column', 'Type', 'Description', 'Valid values', 'Action on invalid', 'Deletion criteria'])

spend = pd.read_parquet(DATA_PROCESSED / 'spend_clean.parquet')
nulls_spend = spend.isna().sum().reset_index()
nulls_spend.columns = ['Column', 'Nulls']
nulls_spend['Nulls %'] = (nulls_spend['Nulls'] / len(spend) * 100).round(1)

spend_dict = spend_dict.merge(nulls_spend, on='Column', how='left')
spend_dict

,Column,Type,Description,Valid values,Action on invalid,Deletion criteria,Nulls,Nulls %
0,Date,datetime,Дата расхода рекламного бюджета,дата в диапазоне 03.07.2023 – 21.06.2024,не встречалось NaN,не удаляем,0,0.0
1,Source,string,Платный канал привлечения лидов,"14 значений: Facebook Ads, Tiktok Ads, Google ...",не встречалось NaN,не удаляем,0,0.0
2,Campaign,string,"Кампания, на которую были израсходованы средства",50 уникальных значений (имена кампаний),NaN сохраняем (29% — каналы без кампанийной ст...,не удаляем,5077,25.6
3,Impressions,int,Количество показов рекламы в указанную дату,≥ 0; 27% записей = 0 (паузы кампаний),не встречалось NaN,не удаляем,0,0.0
4,Spend,float (EUR),Расход на рекламу в евро,≥ 0; общий бюджет за период €149 523,не встречалось NaN,не удаляем,0,0.0
5,Clicks,int,Количество уникальных переходов по рекламной с...,≥ 0,не встречалось NaN,не удаляем,0,0.0
6,AdGroup,string,Название группы объявлений (только для платных...,24 уникальных значения,NaN сохраняем (33% — каналы без AdGroup структ...,не удаляем,5911,29.8
7,Ad,string,Название рекламного объявления,176 уникальных значений,NaN сохраняем (33%),не удаляем,5911,29.8


In [10]:
# output_path = REPORTS / 'data_dictionary.xlsx'

# with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
#     deals_dict.to_excel(writer, sheet_name='Deals', index=False)
#     calls_dict.to_excel(writer, sheet_name='Calls', index=False)
#     spend_dict.to_excel(writer, sheet_name='Spend', index=False)

# print(f"✅ Сохранено: {output_path}")